# NEWPOS4 — seuils et coûts, CPU uniquement

Analyse exploratoire sur 80 événements de validation. Aucun GPU, fit, inférence ou pixel2021. Réutilise les scores sauvegardés et AP exactes archivées. Les 15 seuils sont verrouillés avant toute estimation des scénarios. Aucun choix automatique du scénario gagnant.


In [ ]:
RUN_ANALYSIS = False
import os,sys
if 'torch' in sys.modules: raise RuntimeError('Redémarrer le noyau avant de masquer CUDA')
os.environ['CUDA_VISIBLE_DEVICES']=''
os.environ['OMP_NUM_THREADS']='2'
os.environ['MKL_NUM_THREADS']='2'


In [ ]:
from pathlib import Path
import base64,hashlib,io,zipfile,json
DATA=base64.b64decode('...')
assert hashlib.sha256(DATA).hexdigest()=='80b111f10afd054235502a8b2eefb887ee0470017736a055e21781622cc88ea3'
ROOT=Path('/kaggle/working/geofirex-newpos4-cpu-code')
ROOT.mkdir(exist_ok=False)
with zipfile.ZipFile(io.BytesIO(DATA)) as z:
    for name in z.namelist():
        assert (ROOT/name).resolve().is_relative_to(ROOT.resolve())
    z.extractall(ROOT)
sys.path.insert(0,str(ROOT/'src'))
from geofirex.newpos4_analysis import ResourceGuard,analysis
from geofirex import c8_experiments as c
PINS=c.read(ROOT/'reports/newpos4_thresholds_20261121_v1/analysis_inputs.json')
c.old.verify_manifest(ROOT,PINS['files'])
print(ResourceGuard('/kaggle/working').receipt())


Attacher les sorties existantes du notebook NEWPOS4 comme entrée, avec son archive COMPLETE intacte. Aucune des11 partitions de rasters et aucun checkpoint séparé ne sont nécessaires. Si un upload privé décompresse le ZIP, conserver uniquement une copie de transport .bin aux mêmes octets/SHA ; préserver le ZIP original. Ne jamais télécharger ou régénérer les prédictions par inférence.


In [ ]:
ARCHIVES=[p for p in Path('/kaggle/input').rglob('B2_C8_V2_NEWPOS4_seed42-COMPLETE-*') if p.suffix in ('.zip','.bin') and p.stat().st_size==PINS['source_archive_bytes']]
OUT=Path('/kaggle/working/NEWPOS4_V2_THRESHOLDS_COSTS_CPU_v1')
if RUN_ANALYSIS:
    assert len(ARCHIVES)==1, 'Attacher exactement une archive COMPLETE intacte du run NEWPOS4'
    analysis(ROOT,ARCHIVES[0],OUT)
else: print('Analyse désactivée ; aucun GPU, entraînement ou inférence')


In [ ]:
from IPython.display import display,FileLink
if RUN_ANALYSIS:
    result=c.read(OUT/'results.json')
    for row in result['scenarios']:
        print(row['scenario'],row['summary'],row['threshold_operating_gates'])
    print('Exploratoire validation ; aucune promotion ni sélection automatique')
    for p in sorted(Path('/kaggle/working').glob('NEWPOS4_V2_THRESHOLDS_COSTS_CPU_v1-*.zip')):
        print(c.old.verify_zip(p));display(FileLink(str(p)));display(FileLink(str(p.with_suffix('.sha256'))))
